# Task 1 — Rating prediction (latent factor model)

Explore and pick hyperparameters here; copy the winner into `src/assignment_1/tasks/task1.py`.

HW3 target: public MSE ≤ 0.7280 (baseline 0.7429).

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
from assignment_1.data import load_csv, random_split
from assignment_1.metrics import mse

from assignment_1.models.lfm import fit_svd, predict_svd

DATA_DIR = "../data"  # notebook runs from notebooks/

## Look at the data
Check the rating range: Surprise clips predictions to it, and the baseline assumed 1–5.

In [ ]:
train = load_csv("tasks_1_2_train.csv", DATA_DIR)
test = load_csv("task_1_test.csv", DATA_DIR)
print(
    f"{len(train):,} ratings | {train.user.nunique():,} users | {train.item.nunique():,} items"
)
print(
    f"test users unseen in train: {(~test.user.isin(train.user)).mean():.1%}, items: {(~test.item.isin(train.item)).mean():.1%}"
)
train.rating.value_counts().sort_index()

## Validation split + baseline yardstick
Same split for every model, so differences come from the model, not the split. `rating_scale` comes from the *full* training data so validation-split models clip the same way the final model does.

In [ ]:
tr, val = random_split(train, val_frac=0.25, seed=0)
SCALE = (train.rating.min(), train.rating.max())


def val_mse(**params):
    model = fit_svd(tr, rating_scale=SCALE, **params)
    return mse(val.rating, predict_svd(model, val.user, val.item))


base = val_mse(n_factors=1, n_epochs=50)
print(f"baseline val MSE = {base:.4f}  ->  2% target ≈ {base * 0.98:.4f}")

## Step 1: sweep regularization only (λ = `reg_all`)
If the best value is at the edge of the range, extend the range.

In [ ]:
regs = [0.02, 0.05, 0.1, 0.2, 0.3, 0.5]
scores = [val_mse(n_factors=1, n_epochs=50, reg_all=r) for r in regs]
for r, s in zip(regs, scores):
    print(f"reg_all={r:<5} val MSE={s:.4f}  ({(1 - s / base) * 100:+.1f}% vs baseline)")

plt.plot(regs, scores, "o-")
plt.axhline(base * 0.98, ls="--", label="2% target")
plt.xscale("log")
plt.xlabel("reg_all (λ)")
plt.ylabel("val MSE")
plt.legend()
plt.show()

## Next
- Sweep `n_factors`, `lr_all`, `n_epochs` around the best λ.
- Copy the winner into `TASK1_SVD_PARAMS` in `tasks/task1.py`, then `make predict` (or `make submit`).